# WePocketModel 1.0 4B — generation, embeddings, and decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/WePocketModel-4B-embed-vqa-decisions.ipynb)

[`tencent/WePocketModel1.0-4B`](https://huggingface.co/tencent/WePocketModel1.0-4B) is WeChat AI (Tencent)'s on-device vision-language model (Apache-2.0). **One shared backbone and vision encoder** cover three capabilities:

| Capability | What it returns |
| --- | --- |
| **Generation** | Text from text, images, or interleaved text and images (OCR, captioning, VQA) |
| **Embedding** | L2-normalized text/image vectors. 4B full dim **2048**, with MRL at 64 / 128 / 256 / 512 / 1024 / 2048 |
| **Decision** | Structured `choice`, ordinal `score`, or binary `noul` (yes/no) from application state |

The Hub architecture is Qwen3.5 (`Qwen3_5ForConditionalGeneration`, ~4.54B parameters). GitHub reports about **8 GB / ≈8.5 GiB of BF16 weight memory**; Hub shards plus `unimodel_aux.safetensors` are about **9.1 GB** on disk. Runtime code is [Tencent/WePocketModel](https://github.com/Tencent/WePocketModel) (`UniModelTransformers` — not a raw `transformers.generate` loop). A [2B safetensors](https://huggingface.co/tencent/WePocketModel1.0-2B) sibling and [4B GGUF](https://huggingface.co/tencent/WePocketModel1.0-4B-GGUF) exist.

**Why this model.** One Apache-2.0 checkpoint that can read a page, embed it for retrieval, and return a typed next-action decision without loading three specialist models. GitHub's comparison is about resident weight memory for that combined job, not a T4 benchmark.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/WePocketModel-4B-embed-vqa-decisions.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0 (inference code and the weights in this Hub repo).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM and **no native bfloat16** (`sm_75`). GitHub's CUDA path is **BF16** and recommends **16 GiB**. This notebook loads **float16**. `config.json` records `transformers_version: 5.3.0`; GitHub pins the same. **Do not reinstall torch** (Colab already has a CUDA wheel). This notebook keeps Colab's torch, **pins transformers to 5.3.0**, clones the WePM runtime, and restarts the session. If 4B does not fit, it falls back to the 2B checkpoint and says so.

**Not in this notebook:** GGUF / CPU inference, the 2B checkpoint as the default, vLLM, or copying GitHub's evaluation / HCD numbers.

**Links:** [model card](https://huggingface.co/tencent/WePocketModel1.0-4B) · [GitHub runtime](https://github.com/Tencent/WePocketModel) · [technical report](https://arxiv.org/abs/2611.27365) · [announcement](https://x.com/TencentAI_News/status/2125152448218034505) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/WePocketModel-4B-embed-vqa-decisions.md) · [collection](https://huggingface.co/collections/tencent/wepocketmodel) · [4B GGUF](https://huggingface.co/tencent/WePocketModel1.0-4B-GGUF) · [2B](https://huggingface.co/tencent/WePocketModel1.0-2B)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; load will use float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install (keep Colab's torch)

The [GitHub Quick Start](https://github.com/Tencent/WePocketModel#quick-start) install is:

```bash
git clone https://github.com/Tencent/WePocketModel
cd WePocketModel
python -m pip install torch==2.10.0 torchvision==0.25.0 \
  --index-url https://download.pytorch.org/whl/cu128
python -m pip install -r inference/requirements.txt
```

`inference/requirements.txt` and `config.json` pin **`transformers==5.3.0`**. This notebook pins that release so the WePM runtime (`Qwen3_5ForConditionalGeneration`, SDPA, MGVB) matches the published checkpoint. Colab's preinstalled 5.2x will be **downgraded** — that is intentional. Companion pins: **`huggingface_hub>=1.3.0,<2`**, **`accelerate==1.14.0`** (the WePM requirement), `safetensors`, `jinja2>=3.1.6`, `pillow`.

**Do not reinstall torch** — Colab already ships a CUDA wheel; forcing `torch==2.10.0` from the cu128 index often drops GPU support. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch.

The cell also clones [Tencent/WePocketModel](https://github.com/Tencent/WePocketModel) into `/content/WePocketModel` (or `./WePocketModel` outside Colab). Inference imports (`runtime.py`, `checkpoint.py`) live under `inference/`.

**Do not import the model in this cell.** Colab has already imported its preinstalled transformers. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

INSTALL_TAG = "tf==5.3.0-wepm-4b"
CONTENT = Path("/content") if Path("/content").is_dir() else Path(".")
MARKER = CONTENT / ".wepm_4b_nb_install_ok"
WEPM_DIR = CONTENT / "WePocketModel"
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

# GitHub / config.json: transformers==5.3.0. Keep Colab's CUDA torch.
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers==5.3.0" "huggingface_hub>=1.3.0,<2" "accelerate==1.14.0" "safetensors>=0.5" "jinja2>=3.1.6" pillow',
)

if not (WEPM_DIR / "inference" / "runtime.py").is_file():
    print("cloning https://github.com/Tencent/WePocketModel ->", WEPM_DIR)
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/Tencent/WePocketModel.git",
            str(WEPM_DIR),
        ]
    )
else:
    print("WePM checkout already present:", WEPM_DIR)

print()
print("transformers", md.version("transformers"))
print("huggingface_hub", md.version("huggingface_hub"))
print("accelerate", md.version("accelerate"))
print("safetensors", md.version("safetensors"))
print("jinja2", md.version("jinja2"))
print("WePM dir", WEPM_DIR, "runtime.py", (WEPM_DIR / "inference" / "runtime.py").is_file())

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so the installed transformers is imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you** so the **5.3.0** pin replaces Colab's already-imported transformers. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers==5.3.0`, CUDA, device name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **float16** is what this notebook loads. Put the WePM `inference/` package on `sys.path`.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

CONTENT = Path("/content") if Path("/content").is_dir() else Path(".")
WEPM_DIR = CONTENT / "WePocketModel"
INFER_DIR = WEPM_DIR / "inference"
assert (INFER_DIR / "runtime.py").is_file(), (
    f"WePM runtime not found at {INFER_DIR}. Rerun the install cell."
)
sys.path.insert(0, str(INFER_DIR))

import torch
import transformers

loaded = transformers.__version__.split("+")[0]
assert loaded == "5.3.0", (
    f"transformers {transformers.__version__} is loaded; WePocketModel needs "
    "transformers==5.3.0 (config.json transformers_version 5.3.0; GitHub pin). "
    "Run the install cell, then Runtime > Restart session."
)

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda:0"
# T4 (sm_75) has no native bfloat16. WePM CUDA default is bfloat16.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
PRIMARY_ID = "tencent/WePocketModel1.0-4B"
FALLBACK_ID = "tencent/WePocketModel1.0-2B"

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (WePM CUDA default is bfloat16; T4 has no native bf16)")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")
print("WePM inference path:", INFER_DIR)


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load the model

The published CUDA loader is `UniModelTransformers` in [`inference/runtime.py`](https://github.com/Tencent/WePocketModel/blob/main/inference/runtime.py). It loads `Qwen3_5ForConditionalGeneration` with **SDPA**, then the MGVB module, embedding head, and decision head from `unimodel_aux.safetensors`.

GitHub hardcodes `self.dtype = torch.bfloat16` on CUDA. A T4 cannot run that. The cell below intercepts that assignment so `from_pretrained(..., dtype=...)` gets **float16** on Turing, and the auxiliary slots/heads/MGVB follow.

First run downloads **~9.1 GB** (five backbone shards + 48 MB aux). GitHub waits for checkpoint-size + 2 GiB free (~11 GiB) before load; we pass `min_free_gib` so a 15 GB T4 can start. If 4B still OOM or fails the free-memory check, the cell **falls back to** [`tencent/WePocketModel1.0-2B`](https://huggingface.co/tencent/WePocketModel1.0-2B) (~4.47 GB of BF16 weights) and prints that it did so.


In [ ]:
import gc
import time

from model_source import resolve_model_path
from runtime import UniModelTransformers

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# WePM CUDA init assigns self.dtype = torch.bfloat16. Redirect that to DTYPE
# *before* from_pretrained so weights are not loaded twice.
def _dtype_setattr(self, name, value):
    if name == "dtype" and value is torch.bfloat16:
        value = DTYPE
    object.__setattr__(self, name, value)


UniModelTransformers.__setattr__ = _dtype_setattr


def load_wepm(model_id, min_free_gib):
    print(f"resolving {model_id} …")
    model_dir = resolve_model_path(model_id)
    print("snapshot:", model_dir)
    return UniModelTransformers(
        model_dir,
        device=DEVICE,
        min_free_gib=min_free_gib,
        trust_model_code=False,
    )


USED_FALLBACK = False
MODEL_ID = PRIMARY_ID
model = None
t0 = time.perf_counter()
try:
    # 4B weights ≈8.5 GiB; require ~9 GiB free rather than size+2 (~11 GiB).
    model = load_wepm(PRIMARY_ID, min_free_gib=9.0)
except (RuntimeError, torch.cuda.OutOfMemoryError) as exc:
    print("4B load failed:", type(exc).__name__, exc)
    print("Falling back to the 2B checkpoint:", FALLBACK_ID)
    USED_FALLBACK = True
    MODEL_ID = FALLBACK_ID
    del model
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    model = load_wepm(FALLBACK_ID, min_free_gib=5.0)

torch.cuda.synchronize()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.1f}s")
if USED_FALLBACK:
    print("NOTE: 4B did not fit this runtime; demos below use the 2B checkpoint.")
print("class:", type(model).__name__)
print("dtype:", model.dtype, "| device:", model.device)
print("embedding dim:", model.config.get("unimodel_embedding_dim"))
print("MRL dims:", model.config.get("unimodel_embedding_mrl_dims"))
print("architecture:", model.config.get("unimodel_architecture"))
print_vram("after load")


## 4. Demo 1 — generation (OCR / VQA)

Download a public SROIE receipt from [`hf-internal-testing/fixtures_ocr`](https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr) (same JPEG several OCR notebooks in this repo use). Ask the model to read the merchant, date, and total, then answer a short visual question.

Generation is greedy (`max_new_tokens=96`). The runtime compresses visual tokens with MGVB (budget 512) before decoding.


In [ ]:
import urllib.request
from pathlib import Path

from IPython.display import display
from PIL import Image

SAMPLES = CONTENT / "wepm_samples"
SAMPLES.mkdir(parents=True, exist_ok=True)

RECEIPT_URL = (
    "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/"
    "resolve/main/SROIE-receipt.jpeg"
)
CATS_URLS = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg",
    "http://images.cocodataset.org/val2017/000000039769.jpg",
]
RECEIPT_PATH = SAMPLES / "sroie-receipt.jpeg"
CATS_PATH = SAMPLES / "cats.jpeg"


def download(url, dest, timeout=60):
    dest = Path(dest)
    if dest.is_file() and dest.stat().st_size > 0:
        return dest
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; donvito-notebooks/wepm)"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        dest.write_bytes(resp.read())
    return dest


download(RECEIPT_URL, RECEIPT_PATH)
print("receipt:", RECEIPT_PATH, RECEIPT_PATH.stat().st_size, "bytes")
receipt = Image.open(RECEIPT_PATH).convert("RGB")
display(receipt.resize((360, int(360 * receipt.height / receipt.width))))

gen = model.generate(
    [
        {"type": "image", "path": RECEIPT_PATH},
        {
            "type": "text",
            "text": (
                "Read the merchant name, date, and total amount from this receipt. "
                "Then answer: is this a restaurant bill or a grocery/store receipt? "
                "Reply in a few short lines."
            ),
        },
    ],
    max_new_tokens=96,
)
print(gen["text"])
print("finish_reason:", gen["finish_reason"], f"| {gen['seconds']:.1f}s")
print_vram("after generation")


## 5. Demo 2 — embedding (rank texts and images)

Embed a query, two images, and a few short texts. Vectors are L2-normalized, so cosine similarity is a dot product. The 4B full dimension is **2048**; the cell also prints a **256-d MRL prefix** (the runtime L2-normalizes the selected prefix, as on the card).


In [ ]:
import numpy as np

cats_ok = False
for url in CATS_URLS:
    try:
        download(url, CATS_PATH)
        cats_ok = CATS_PATH.stat().st_size > 0
        print("cats image:", url)
        break
    except Exception as exc:
        print("cats download failed:", url, type(exc).__name__, exc)
if cats_ok:
    cats = Image.open(CATS_PATH).convert("RGB")
    display(cats.resize((320, int(320 * cats.height / cats.width))))
else:
    print("Skipping the cat image; ranking the remaining candidates.")

query_text = "a printed store receipt with line items and a total"
candidates = [
    ("receipt image", "image", RECEIPT_PATH),
    ("receipt caption", "text", "A supermarket receipt with prices and a total at the bottom."),
    ("unrelated text", "text", "A hiking trail through alpine mountains at sunset."),
]
if cats_ok:
    candidates.insert(1, ("cats image", "image", CATS_PATH))
    candidates.append(("cats caption", "text", "A fluffy cat sitting in a field of flowers."))

full_dim = int(model.config["unimodel_embedding_dim"])
mrl = [int(d) for d in model.config["unimodel_embedding_mrl_dims"]]
prefix_dim = 256 if 256 in mrl else mrl[0]
EMBED_DIMS = [full_dim, prefix_dim]
print("embedding dims:", EMBED_DIMS, "(full + MRL prefix)")


def embed_item(kind, value, dims=EMBED_DIMS):
    if kind == "image":
        content = [{"type": "image", "path": value}]
    else:
        content = [{"type": "text", "text": value}]
    out = model.embed(content, dims=dims)
    return {int(k): np.asarray(v, dtype=np.float32) for k, v in out["embeddings"].items()}


query_vecs = embed_item("text", query_text)
print(f"query: {query_text!r}")
print(f"query {full_dim}-d L2 norm:", float(np.linalg.norm(query_vecs[full_dim])))

print()
print(f"{'candidate':<18} {f'cos@{full_dim}':>10} {f'cos@{prefix_dim}':>10}")
ranked = []
for name, kind, value in candidates:
    vecs = embed_item(kind, value)
    scores = {dim: float(query_vecs[dim] @ vecs[dim]) for dim in EMBED_DIMS}
    ranked.append((scores[full_dim], name, scores))
    print(f"{name:<18} {scores[full_dim]:10.3f} {scores[prefix_dim]:10.3f}")

print()
print(f"rank by {full_dim}-d cosine (higher = closer to the receipt query):")
for i, (score, name, _) in enumerate(sorted(ranked, reverse=True), start=1):
    print(f"  {i}. {name}  ({score:.3f})")
print_vram("after embedding")


## 6. Demo 3 — decision (choice, score, binary)

Decision uses a separate learned J slot and a direct candidate-index head. The request format is the card's SystemOne-visible JSON: `state` plus `questions.q` with `type` `choice`, `score`, or `noul` (binary). The runtime returns probabilities over the listed candidates — not generated tokens.

These are smoke examples of the wire format, not a claim that the answers match a production policy. The card: evaluate output quality on representative inputs and **validate structured decisions before using them to trigger actions**.


In [ ]:
from pprint import pprint

requests = [
    {
        "label": "choice — support reply",
        "request": {
            "state": "A customer received the wrong item and asks for a replacement.",
            "questions": {
                "q": {
                    "type": "choice",
                    "instructions": "Choose the response that best addresses the customer's request.",
                    "criteria": {
                        "replace": "Apologize and arrange a replacement for the correct item.",
                        "ignore": "Ignore the request and close the conversation.",
                        "refuse": "Tell the customer that no help is available.",
                    },
                }
            },
        },
    },
    {
        "label": "score — urgency 1–5",
        "request": {
            "state": "The fridge arrived today. The door will not close and food is already warming.",
            "questions": {
                "q": {
                    "type": "score",
                    "instructions": "Score how urgently a human agent should take this ticket.",
                    "criteria": [
                        "1 lowest urgency",
                        "2 low",
                        "3 medium",
                        "4 high",
                        "5 highest urgency",
                    ],
                }
            },
        },
    },
    {
        "label": "noul — binary (image in state)",
        "request": {
            "state": "Does this photograph show a printed store receipt with prices?",
            "questions": {
                "q": {
                    "type": "noul",
                    "instructions": "Answer yes if the image is a receipt, otherwise no.",
                }
            },
        },
        "images": [RECEIPT_PATH],
    },
]

for item in requests:
    print("=" * 72)
    print(item["label"])
    images = [str(p) for p in item.get("images") or []]
    result = model.decide(item["request"], images)
    pprint(result["answers"])
    print(f"{result['seconds']:.2f}s | candidates: {result['candidate_count']}")

print_vram("after decision")


## 7. Caveats and cleanup

**What this notebook is for.** The WePM Transformers path on a free T4: OCR/VQA generation, L2-normalized multimodal embeddings with an MRL prefix, and typed `choice` / `score` / `noul` decisions. `torch.cuda.max_memory_allocated` is printed after load, after each demo, and again below.

**What it is not.** GGUF / CPU (`tencent/WePocketModel1.0-4B-GGUF`), a raw Hugging Face `generate` chat loop, or the 2B checkpoint unless 4B failed to load. GitHub's evaluation figure and HCD comparison are project-reported; they are not rerun here.

**Transformers pin.** GitHub and `config.json` use **`transformers==5.3.0`**. The install cell downgrades Colab's newer transformers and restarts. Do not keep Colab's preinstalled 5.2x.

**T4 / dtype.** GitHub CUDA inference is BF16 (16 GiB recommended). A T4 has no native bf16. The load cell intercepts `self.dtype = torch.bfloat16` so weights, MGVB, and task heads load in **float16**. Full float32 (~17 GiB of weights) will not fit.

**VRAM.** 4B download is ~9.1 GB; GitHub lists ≈8.5 GiB of BF16 weights. Peak during these short demos is expected around **11–13 GB** on 15 GB — an estimate from the weight size and the 16 GiB CUDA guidance, **not a measured Colab log attached to this file**. The cell below prints the real peak when you run it. If 4B did not fit, the load cell already said so and the demos used 2B.

**Decision outputs.** Probabilities over listed candidates. Validate them before they trigger actions (card / GitHub intended-use note).

**Related notebooks:** [WeMM-Video-2B](WeMM-Video-2B-audio-visual-video-understanding.ipynb) (audio-visual video, text out), [WeVisOne-VL 2B](WeVisOne-VL-2B-detect-segment.ipynb) (boxes / cutouts), [MiniCPM-V 4.7 1B](MiniCPM-V-4.7-1B-vision.ipynb), [d1-2B decisions](d1-2B-inference-decisions.ipynb), [Nemotron 3.5 Embed 1B](Nemotron-3.5-Embed-1B-visual-rag.ipynb), [EmbeddingGemma 2](embeddinggemma-2-inference-similarity.ipynb).

Free the GPU before you open another notebook:


In [ ]:
import gc

print("checkpoint:", MODEL_ID, "| fallback:", USED_FALLBACK)
print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del model
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
print(f"torch.cuda.max_memory_allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
